## CHURN PREDICTION using Binary Classification

In [1]:
%pip install -q kagglehub pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import kagglehub

path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print("Downloaded to:", path)

Downloaded to: /home/codespace/.cache/kagglehub/datasets/blastchar/telco-customer-churn/versions/1


In [3]:
import pandas as pd

df = pd.read_csv(f"{path}/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
df.head().T # Transpose the first 5 rows of the DataFrame to view them in a vertical format.

,0,1,2,3,4
customerID,7590-VHVEG,5575-GNVDE,3668-QPYBK,7795-CFOCW,9237-HQITU
gender,Female,Male,Male,Male,Female
SeniorCitizen,0,0,0,0,0
Partner,Yes,No,No,No,No
Dependents,No,No,No,No,No
tenure,1,34,2,45,2
PhoneService,No,Yes,Yes,No,Yes
MultipleLines,No phone service,No,No,No phone service,No
InternetService,DSL,DSL,DSL,DSL,Fiber optic
OnlineSecurity,No,Yes,Yes,Yes,No


In [5]:
# Clean up the column names
df.columns = df.columns.str.lower().str.replace(' ','_')

categorical_variables = list(df.dtypes[(df.dtypes == 'str')].index)

# Clean up the body of the table
for c in categorical_variables: 
    df[c] = df[c].str.lower().str.replace(' ','_')

df.head().T

,0,1,2,3,4
customerid,7590-vhveg,5575-gnvde,3668-qpybk,7795-cfocw,9237-hqitu
gender,female,male,male,male,female
seniorcitizen,0,0,0,0,0
partner,yes,no,no,no,no
dependents,no,no,no,no,no
tenure,1,34,2,45,2
phoneservice,no,yes,yes,no,yes
multiplelines,no_phone_service,no,no,no_phone_service,no
internetservice,dsl,dsl,dsl,dsl,fiber_optic
onlinesecurity,no,yes,yes,yes,no


In [6]:
df.dtypes

customerid              str
gender                  str
seniorcitizen         int64
partner                 str
dependents              str
tenure                int64
phoneservice            str
multiplelines           str
internetservice         str
onlinesecurity          str
onlinebackup            str
deviceprotection        str
techsupport             str
streamingtv             str
streamingmovies         str
contract                str
paperlessbilling        str
paymentmethod           str
monthlycharges      float64
totalcharges            str
churn                   str
dtype: object

In [7]:
tc = pd.to_numeric(df['totalcharges'],errors="coerce")
tc

0         29.85
1       1889.50
2        108.15
3       1840.75
4        151.65
         ...   
7038    1990.50
7039    7362.90
7040     346.45
7041     306.60
7042    6844.50
Name: totalcharges, Length: 7043, dtype: float64

In [8]:
df['totalcharges'] = pd.to_numeric(df['totalcharges'],errors="coerce")
df['totalcharges'] = df['totalcharges'].fillna(0)

In [9]:
df[tc.isnull()][['customerid','totalcharges']]

,customerid,totalcharges
488,4472-lvygi,0.0
753,3115-czmzd,0.0
936,5709-lvoeq,0.0
1082,4367-nuyao,0.0
1340,1371-dwpaz,0.0
3331,7644-omvmy,0.0
3826,3213-vvolg,0.0
4380,2520-sgtta,0.0
5218,2923-arzlg,0.0
6670,4075-wkniu,0.0


In [10]:
df.churn = (df.churn == 'yes').astype(int)

## Validation Framework
Performing the train/val/test with scikit-learn

In [11]:
from sklearn.model_selection import train_test_split

In [12]:
df_full_train, df_test = train_test_split(df,test_size=0.2,random_state=1)
len(df_test), len(df_full_train)

df_train, df_val = train_test_split(df_full_train,test_size=0.25,random_state=1)
len(df_test), len(df_train), len(df_val)

(1409, 4225, 1409)

In [13]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [14]:
y_train = df_train.churn.values 
y_val = df_val.churn.values 
y_test = df_test.churn.values 

df_train = df_train.drop(columns=['churn'])
df_val = df_val.drop(columns=['churn'])
df_test = df_test.drop(columns=['churn'])

In [15]:
df_train.T

,0,1,2,3,4,5,6,7,8,9,...,4215,4216,4217,4218,4219,4220,4221,4222,4223,4224
customerid,8015-ihcgw,1960-uycnn,9250-wypll,6786-obwqr,1328-euzhc,8676-ooqej,1452-voqch,6653-cbbom,5893-kclgt,3992-ywpko,...,3190-fzatl,7734-dboai,0369-zgovk,6937-gcdgq,3589-ppvkw,1309-xgfsn,4819-hjpiw,3703-vavcl,3812-lrzir,1704-nrwye
gender,female,male,female,female,female,male,male,female,female,female,...,male,female,female,male,male,male,male,male,female,female
seniorcitizen,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,1
partner,yes,no,no,yes,yes,no,no,no,no,no,...,no,yes,yes,yes,no,yes,no,yes,yes,no
dependents,yes,no,no,yes,no,no,no,no,yes,no,...,yes,yes,yes,yes,no,yes,no,yes,yes,no
tenure,72,10,5,5,18,4,1,1,72,6,...,14,1,28,19,9,52,18,2,27,9
phoneservice,yes,yes,yes,yes,yes,no,yes,yes,yes,yes,...,yes,no,yes,yes,yes,yes,no,yes,yes,yes
multiplelines,yes,yes,yes,no,no,no_phone_service,yes,no,no,yes,...,no,no_phone_service,no,no,no,yes,no_phone_service,no,yes,no
internetservice,fiber_optic,fiber_optic,fiber_optic,fiber_optic,no,dsl,fiber_optic,fiber_optic,no,fiber_optic,...,no,dsl,fiber_optic,dsl,dsl,dsl,dsl,fiber_optic,no,fiber_optic
onlinesecurity,yes,no,no,no,no_internet_service,no,no,no,no_internet_service,no,...,no_internet_service,no,no,yes,no,no,no,no,no_internet_service,no


## Exploratory Data Analysis
1. check missing values
2. look at the target variable (churn)
3. look at the numerical and categorical variables

In [16]:
df_full_train = df_full_train.reset_index(drop=True)
df_full_train.head()

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,5442-pptjy,male,0,yes,yes,12,yes,no,no,no_internet_service,...,no_internet_service,no_internet_service,no_internet_service,no_internet_service,two_year,no,mailed_check,19.70,258.35,0
1,6261-rcvns,female,0,no,no,42,yes,no,dsl,yes,...,yes,yes,no,yes,one_year,no,credit_card_(automatic),73.90,3160.55,1
2,2176-osjuv,male,0,yes,no,71,yes,yes,dsl,yes,...,no,yes,no,no,two_year,no,bank_transfer_(automatic),65.15,4681.75,0
3,6161-erdgd,male,0,yes,yes,71,yes,yes,dsl,yes,...,yes,yes,yes,yes,one_year,no,electronic_check,85.45,6300.85,0
4,2364-ufrom,male,0,no,no,30,yes,no,dsl,yes,...,no,yes,yes,no,one_year,no,electronic_check,70.40,2044.75,0


In [17]:
df_full_train.isnull().sum() ## no missing values found

customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
churn               0
dtype: int64

In [18]:
df_full_train.churn.value_counts() #shows the value counts in the column
## Churn Rate
df_full_train.churn.value_counts(normalize=True) #shows the value counts as a percentage

churn
0    0.730032
1    0.269968
Name: proportion, dtype: float64

In [19]:
## for a column with binary variable, you can also just calculate the mean 
global_churn_rate = df_full_train.churn.mean()
round(global_churn_rate, 2)

np.float64(0.27)

In [20]:
list(df_full_train.columns)

['customerid',
 'gender',
 'seniorcitizen',
 'partner',
 'dependents',
 'tenure',
 'phoneservice',
 'multiplelines',
 'internetservice',
 'onlinesecurity',
 'onlinebackup',
 'deviceprotection',
 'techsupport',
 'streamingtv',
 'streamingmovies',
 'contract',
 'paperlessbilling',
 'paymentmethod',
 'monthlycharges',
 'totalcharges',
 'churn']

In [21]:
numerical = [
    'tenure', 
    'monthlycharges', 
    'totalcharges'
    ]
categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod'
]
#df_full_train.dtypes
df_full_train[categorical].nunique()


gender              2
seniorcitizen       2
partner             2
dependents          2
phoneservice        2
multiplelines       3
internetservice     3
onlinesecurity      3
onlinebackup        3
deviceprotection    3
techsupport         3
streamingtv         3
streamingmovies     3
contract            3
paperlessbilling    2
paymentmethod       4
dtype: int64

## Feature Importance: churn rate & risk ratio
feature important analysis, determines which features affect our target variable

In [22]:
df_full_train.head()
churn_female = df_full_train[df_full_train['gender'] == 'female'].churn.mean()
churn_female

np.float64(0.27682403433476394)

In [23]:
churn_male = df_full_train[df_full_train['gender'] == 'male'].churn.mean()
churn_male, global_churn_rate

(np.float64(0.2632135306553911), np.float64(0.26996805111821087))

In [24]:
df_full_train.partner.value_counts()

partner
no     2932
yes    2702
Name: count, dtype: int64

In [25]:
churn_partner = df_full_train[df_full_train['partner'] == 'yes'].churn.mean()
churn_no_partner = df_full_train[df_full_train['partner'] == 'no'].churn.mean()
churn_partner, churn_no_partner
global_churn_rate-churn_partner, global_churn_rate-churn_no_partner

(np.float64(0.06493474245795922), np.float64(-0.05984095297455855))

In [26]:
churn_no_partner / global_churn_rate, churn_partner / global_churn_rate

(np.float64(1.2216593879412643), np.float64(0.7594724924338315))

In [27]:
# df_full_train is a DataFrame
# groupby(...) creates groups based on one or more columns

# 1) Group by a single column
df_full_train.groupby('contract')['churn'].mean()

# Explanation:
# - 'contract' = the column used to define groups
# - ['churn'] = the column we want to aggregate
# - .mean() = the calculation to apply within each group

# 2) Group by multiple columns
df_full_train.groupby(['contract', 'gender'])['churn'].mean()

# Explanation:
# - creates one group for each unique combination of contract + gender
# - then computes churn mean inside each group

# 3) Group and aggregate multiple metrics
df_full_train.groupby('contract').agg(
    churn_rate=('churn', 'mean'),
    customers=('customerid', 'count'),
    avg_tenure=('tenure', 'mean')
)

# Explanation:
# - .agg(...) lets you compute several summary values at once
# - each tuple is: new_column_name, function
# - here we calculate:
#   - churn_rate = average churn
#   - customers = number of rows in each group
#   - avg_tenure = average tenure

# 4) Group by a column but keep the group key as a regular column
df_full_train.groupby('contract', as_index=False)['churn'].mean()

# Explanation:
# - as_index=False keeps the grouping column in the result as a normal column
# - without it, the group key is usually used as the index

# 5) You can also group and then filter
df_full_train.groupby('contract').filter(lambda g: g['churn'].mean() > 0.3)

# Explanation:
# - groups are formed first
# - then only groups matching the condition remain

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
5,4765-oxppd,female,0,yes,yes,9,yes,no,dsl,yes,...,yes,yes,no,no,month-to-month,no,mailed_check,65.00,663.05,1
7,5213-twwju,male,0,no,no,28,yes,no,fiber_optic,no,...,yes,yes,no,yes,month-to-month,no,electronic_check,91.00,2626.15,0
8,9957-yodkz,male,1,yes,no,6,yes,yes,fiber_optic,no,...,yes,no,no,no,month-to-month,yes,electronic_check,80.80,457.10,0
9,1732-vhubq,female,1,yes,yes,47,yes,no,fiber_optic,no,...,no,no,no,no,month-to-month,no,bank_transfer_(automatic),70.55,3309.25,1
11,7017-vfuly,female,0,yes,no,2,yes,no,no,no_internet_service,...,no_internet_service,no_internet_service,no_internet_service,no_internet_service,month-to-month,no,bank_transfer_(automatic),20.10,43.15,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5624,5345-bmkwb,male,0,yes,no,8,yes,no,no,no_internet_service,...,no_internet_service,no_internet_service,no_internet_service,no_internet_service,month-to-month,no,mailed_check,20.25,158.35,0
5628,2250-ivbwa,male,0,yes,yes,64,yes,no,fiber_optic,yes,...,no,no,no,no,month-to-month,no,electronic_check,81.05,5135.35,0
5629,0781-lkxbr,male,1,no,no,9,yes,yes,fiber_optic,no,...,yes,no,yes,yes,month-to-month,yes,electronic_check,100.50,918.60,1
5631,8868-wozgu,male,0,no,no,28,yes,yes,fiber_optic,no,...,yes,no,yes,yes,month-to-month,yes,electronic_check,105.70,2979.50,1


In [28]:
from IPython import display

In [29]:
for c in categorical: 
        
    print(c)
    df_group = df_full_train.groupby(c)['churn'].agg(['mean','count'])
    df_group['diff'] = df_group['mean'] - global_churn_rate
    df_group['risk'] = df_group['mean'] / global_churn_rate
    display.display(df_group)
    print()
    print()

gender


,mean,count,diff,risk
gender,,,,
female,0.276824,2796,0.006856,1.025396
male,0.263214,2838,-0.006755,0.974980




seniorcitizen


,mean,count,diff,risk
seniorcitizen,,,,
0,0.242270,4722,-0.027698,0.897403
1,0.413377,912,0.143409,1.531208




partner


...

,mean,count,diff,risk
contract,,,,
month-to-month,0.431701,3104,0.161733,1.599082
one_year,0.120573,1186,-0.149395,0.446621
two_year,0.028274,1344,-0.241694,0.104730




paperlessbilling


,mean,count,diff,risk
paperlessbilling,,,,
no,0.172071,2313,-0.097897,0.637375
yes,0.338151,3321,0.068183,1.252560




paymentmethod


,mean,count,diff,risk
paymentmethod,,,,
bank_transfer_(automatic),0.168171,1219,-0.101797,0.622928
credit_card_(automatic),0.164339,1217,-0.105630,0.608733
electronic_check,0.455890,1893,0.185922,1.688682
mailed_check,0.193870,1305,-0.076098,0.718121


## Feature importance: mutual information

Before training a model, we want to know which categorical features carry the most information about churn. **Mutual information (MI)** measures how much knowing a feature reduces our uncertainty about the target.

### Definition

$$I(X;Y) = \sum_{x,y} p(x,y)\,\log\frac{p(x,y)}{p(x)\,p(y)}$$

This compares how often each feature value and churn outcome actually occur together, $p(x,y)$, with how often they would occur together if the feature and churn were independent, $p(x)\,p(y)$.

The same quantity in entropy terms:

$$I(X;Y) = H(Y) - H(Y \mid X)$$

This is our uncertainty about churn before seeing the feature, minus the uncertainty that remains after seeing it.

### Why MI for this dataset

- Most Telco features (`contract`, `internetservice`, `paymentmethod`, ...) are categorical. MI works on them directly, with no encoding needed.
- It captures any kind of dependence, not just linear trends.
- It produces one score per feature, so the features can be ranked.

### Reading the scores

- $I = 0$ means the feature is independent of churn. Higher scores mean the feature is more informative.
- `sklearn.metrics.mutual_info_score` reports values in **nats** (natural log).
- The upper bound is the entropy of churn itself. With a churn rate of about 27%, $H(\text{churn}) \approx 0.58$ nats. So a feature with MI of 0.1 removes roughly a sixth of our uncertainty about churn.
- The scores are most useful for comparing features against each other, not as standalone numbers.

### Caveats

- **Training data only:** MI is computed on the training split so the validation and test sets stay unseen.
- **Categorical features only:** numerical features (`tenure`, `monthlycharges`, `totalcharges`) are assessed separately with correlation.
- **Bias toward many categories:** features with more levels have higher entropy and can score higher partly for that reason.
- **Pairwise, not joint:** MI scores each feature on its own and ignores redundancy between features. Several service columns (`onlinesecurity`, `techsupport`, `streamingtv`, ...) share a "No internet service" level, so part of their scores overlaps with `internetservice`.

In [30]:
from sklearn.metrics import mutual_info_score

mutual_info_score(df_full_train.churn, df_full_train.contract)

0.0983203874041556

In [31]:
mutual_info_score(df_full_train.churn, df_full_train.gender)

0.0001174846211139946

In [32]:
mutual_info_score(df_full_train.churn, df_full_train.partner)

0.009967689095399745

### Mutual information: comparing categorical features with churn

**Objective:** Calculate how much information each categorical feature provides about whether a customer churns.

```python
def mutual_info_churn_score(series):
    return mutual_info_score(df_full_train.churn, series)

df_full_train[categorical].apply(mutual_info_churn_score)
```

### 1. Define a function that compares one feature with churn

```python
def mutual_info_churn_score(series):
    return mutual_info_score(df_full_train.churn, series)
```

- `series` is a placeholder for one complete feature column, such as `contract`.
- `df_full_train.churn` is the target column containing customers’ churn outcomes.
- `mutual_info_score(...)` compares the feature values with the churn outcomes and calculates one numerical score.
- `return` sends that score back.

For example:

```python
mutual_info_churn_score(df_full_train['contract'])
```

Inside the function, `series` now refers to the `contract` column. The calculation becomes:

```python
mutual_info_score(df_full_train.churn, df_full_train['contract'])
```

The two columns contain values for the same customers, in the same order.

### 2. Select the categorical feature columns

```python
df_full_train[categorical]
```

`categorical` is a list of column names. For example:

```python
categorical = ['gender', 'contract', 'paymentmethod']
```

Using that list inside square brackets selects those columns from the DataFrame.

### 3. Apply the function to each selected column

```python
df_full_train[categorical].apply(mutual_info_churn_score)
```

For a DataFrame, `.apply()` works **column by column by default** (`axis=0`).

It passes each complete column into `mutual_info_churn_score` as the `series` argument:

| Column passed in as `series` | Calculation |
|---|---|
| `gender` | Compare gender with churn |
| `contract` | Compare contract with churn |
| `paymentmethod` | Compare payment method with churn |

**It does not calculate a score for each customer. It calculates one score for each feature column.**

### Why does the function name have no parentheses here?

```python
.apply(mutual_info_churn_score)
```

We are handing `.apply()` the function so that it can call the function with each column.

Compare:

```python
# Call the function now with a specific column:
mutual_info_churn_score(df_full_train['contract'])

# Give apply the function to call for every selected column:
df_full_train[categorical].apply(mutual_info_churn_score)
```

### 4. Understand the output

The result is a pandas Series:
- The index contains feature names.
- The values contain their mutual information scores.

Example output — these numbers are illustrative:

```text
gender           0.001
contract         0.098
paymentmethod    0.044
dtype: float64
```

Here, `contract` provides more information about churn than `gender`.

### How to interpret the scores

- **Zero:** No statistical association in the observed data.
- **Higher score:** More information about churn.
- Scores are **not percentages** and are **not restricted to 0–1**.
- A score does not tell us which category increases or decreases churn.
- An association does not prove that a feature causes churn.
- These scores compare each feature with churn individually; they do not measure its contribution within a trained model.

To rank the features from highest score to lowest:

```python
df_full_train[categorical].apply(
    mutual_info_churn_score
).sort_values(ascending=False)
```

### Remember the flow

**Select categorical columns → pass each column into the function → compare it with churn → collect one score per column.**

In [33]:
def mutual_info_churn_score(series): 
    return mutual_info_score(df_full_train.churn, series)

mi = df_full_train[categorical].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)

contract            0.098320
onlinesecurity      0.063085
techsupport         0.061032
internetservice     0.055868
onlinebackup        0.046923
deviceprotection    0.043453
paymentmethod       0.043210
streamingtv         0.031853
streamingmovies     0.031581
paperlessbilling    0.017589
dependents          0.012346
partner             0.009968
seniorcitizen       0.009410
multiplelines       0.000857
phoneservice        0.000229
gender              0.000117
dtype: float64

## Feature importance: correlation

For the numerical features (`tenure`, `monthlycharges`, `totalcharges`), we measure how strongly each one moves with churn using the **Pearson correlation coefficient**.

### Definition

$$r = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_i (x_i - \bar{x})^2}\,\sqrt{\sum_i (y_i - \bar{y})^2}}$$

This is the covariance of the feature and the target, divided by the product of their standard deviations, which keeps it between $-1$ and $1$.

Because churn is coded as 0/1, the formula reduces to the **point-biserial correlation**:

$$r = \frac{\bar{x}_{\text{churn}} - \bar{x}_{\text{no churn}}}{s_x}\sqrt{p(1-p)}$$

Here $s_x$ is the feature's standard deviation and $p$ is the churn rate. In plain terms, $r$ measures how different the feature's average is for churners vs. non-churners, in standard-deviation units, scaled by the class balance.

### Why correlation for these features

- It works directly on numerical values with no binning.
- It gives both **strength** (magnitude) and **direction** (sign) in a single number.
- The sign is easy to read: a negative $r$ for `tenure` would mean customers who have stayed longer churn less.

### Reading the scores

- $r > 0$: higher values of the feature go with **more** churn.
- $r < 0$: higher values of the feature go with **less** churn.
- $r = 0$: no *linear* relationship.
- A rough convention for $|r|$: below 0.1 is negligible, 0.1–0.3 is weak, 0.3–0.5 is moderate, and above 0.5 is strong.

### Caveats

- **Training data only:** correlation is computed on the training split so the validation and test sets stay unseen.
- **Linear only:** Pearson only detects straight-line relationships. Churn tends to fall off sharply in a customer's early months and then flatten, which a single $r$ can understate. Spearman rank correlation (`method='spearman'`) captures any monotonic trend.
- **Sensitive to skew and outliers:** `totalcharges` is heavily right-skewed, so a few large values can pull $r$ around.
- **Redundancy:** `totalcharges` is roughly `tenure × monthlycharges`, so it is strongly correlated with `tenure`. Checking the feature-to-feature correlation matrix shows how much it adds beyond the other two.
- **Not comparable to MI:** correlation and mutual information are on different scales. Use each to rank features *within* its own group, not across groups.

In [34]:
numerical = ['tenure', 'monthlycharges', 'totalcharges']

# correlation of each feature with churn, strongest first
corr = df_full_train[numerical].corrwith(df_full_train.churn)
corr.reindex(corr.abs().sort_values(ascending=False).index)

# feature-to-feature correlation, to check redundancy
df_full_train[numerical].corr()

,tenure,monthlycharges,totalcharges
tenure,1.000000,0.251072,0.828268
monthlycharges,0.251072,1.000000,0.650913
totalcharges,0.828268,0.650913,1.000000


In [35]:
df_full_train.tenure.max()
df_full_train[numerical].corrwith(df_full_train.churn)

tenure           -0.351885
monthlycharges    0.196805
totalcharges     -0.196353
dtype: float64

In [36]:
df_full_train[df_full_train.tenure <= 2].churn.mean(), df_full_train[df_full_train.tenure > 2].churn.mean()

df_full_train[(df_full_train.tenure > 2) & (df_full_train.tenure <= 12)].churn.mean()

np.float64(0.3994413407821229)

## DictVectorizer and One-Hot Encoding

Logistic regression needs numerical inputs. `DictVectorizer` prepares our customer features for the model.

### 1. Convert customer rows into dictionaries

```python
train_dicts = df_train[categorical + numerical].to_dict(orient='records')
```

- `categorical + numerical`: combine the lists of feature column names.
- `orient='records'`: create one dictionary per customer row.
- Keep `churn` separate as the target.

This step changes the data format; categorical values are still text.

### 2. Represent categories using one-hot encoding

Each category seen during fitting gets its own column:

- `1`: the customer has this category.
- `0`: the customer does not have this category.

Example for `contract`:

| Original value | contract=month-to-month | contract=two_year |
|---|---:|---:|
| month-to-month | 1 | 0 |
| two_year | 0 | 1 |

`DictVectorizer` one-hot encodes string values. Numerical values, such as tenure and monthly charges, retain their values.

### 3. Create and fit the converter

```python
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
```

- `sparse=False`: return a regular NumPy array.
- `fit`: learn which features/categories correspond to which columns.
- `transform`: convert the dictionaries into numerical rows.
- `fit_transform`: perform both actions.

Inspect the column names:

```python
dv.get_feature_names_out()
```

### 4. Reuse the converter for validation customers

```python
val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)
```

Use the same fitted converter so feature columns have consistent meanings and order.

### 5. Separate encoding from model training

```python
model.fit(X_train, y_train)
```

**`dv` learns the feature-to-column mapping. `model` learns regression weights using features and actual churn labels.**

The encoded matrix has one row per customer and one column per encoded feature.

In [37]:
from sklearn.feature_extraction import DictVectorizer

In [38]:
## create a dictionary vectorizer object with the variables oriented as records, which is a list of dictionaries where each dictionary represents a row in the DataFrame. 
# This is useful for converting categorical variables into a format suitable for machine learning models.

train_dicts = df_train[categorical + numerical].to_dict(orient = 'Records')
train_dicts[0]

{'gender': 'female',
 'seniorcitizen': 0,
 'partner': 'yes',
 'dependents': 'yes',
 'phoneservice': 'yes',
 'multiplelines': 'yes',
 'internetservice': 'fiber_optic',
 'onlinesecurity': 'yes',
 'onlinebackup': 'yes',
 'deviceprotection': 'yes',
 'techsupport': 'yes',
 'streamingtv': 'yes',
 'streamingmovies': 'yes',
 'contract': 'two_year',
 'paperlessbilling': 'yes',
 'paymentmethod': 'electronic_check',
 'tenure': 72,
 'monthlycharges': 115.5,
 'totalcharges': 8425.15}

In [39]:
dv = DictVectorizer(sparse=False) # sparse=False means we want a dense array as output
# Fit the DictVectorizer to the training data

In [40]:
dv.fit(train_dicts) 
#fit the DictVectorizer to the training data, learning the mapping of feature names to indices in the output array.

DictVectorizer(sparse=False)

In [41]:
dv.fit(train_dicts)

# scikit-learn >= 1.0
print(dv.get_feature_names_out())
# print(dv.get_feature_names_out()) returns the names of the features in the transformed data, 
# which can be useful for understanding the mapping of categorical variables to numerical representations.

# older scikit-learn:
# print(dv.get_feature_names())
# or
# print(dv.feature_names_)

['contract=month-to-month' 'contract=one_year' 'contract=two_year'
 'dependents=no' 'dependents=yes' 'deviceprotection=no'
 'deviceprotection=no_internet_service' 'deviceprotection=yes'
 'gender=female' 'gender=male' 'internetservice=dsl'
 'internetservice=fiber_optic' 'internetservice=no' 'monthlycharges'
 'multiplelines=no' 'multiplelines=no_phone_service' 'multiplelines=yes'
 'onlinebackup=no' 'onlinebackup=no_internet_service' 'onlinebackup=yes'
 'onlinesecurity=no' 'onlinesecurity=no_internet_service'
 'onlinesecurity=yes' 'paperlessbilling=no' 'paperlessbilling=yes'
 'partner=no' 'partner=yes' 'paymentmethod=bank_transfer_(automatic)'
 'paymentmethod=credit_card_(automatic)' 'paymentmethod=electronic_check'
 'paymentmethod=mailed_check' 'phoneservice=no' 'phoneservice=yes'
 'seniorcitizen' 'streamingmovies=no'
 'streamingmovies=no_internet_service' 'streamingmovies=yes'
 'streamingtv=no' 'streamingtv=no_internet_service' 'streamingtv=yes'
 'techsupport=no' 'techsupport=no_interne

In [42]:
dv.transform(train_dicts)
# transform the training data using the fitted DictVectorizer, converting the dictionaries into a numerical array.

array([[0.00000e+00, 0.00000e+00, 1.00000e+00, ..., 1.00000e+00,
        7.20000e+01, 8.42515e+03],
       [1.00000e+00, 0.00000e+00, 0.00000e+00, ..., 0.00000e+00,
        1.00000e+01, 1.02155e+03],
       [1.00000e+00, 0.00000e+00, 0.00000e+00, ..., 0.00000e+00,
        5.00000e+00, 4.13650e+02],
       ...,
       [1.00000e+00, 0.00000e+00, 0.00000e+00, ..., 1.00000e+00,
        2.00000e+00, 1.90050e+02],
       [0.00000e+00, 0.00000e+00, 1.00000e+00, ..., 0.00000e+00,
        2.70000e+01, 7.61950e+02],
       [1.00000e+00, 0.00000e+00, 0.00000e+00, ..., 0.00000e+00,
        9.00000e+00, 7.51650e+02]], shape=(4225, 45))

In [43]:
list((dv.transform(train_dicts)[:5])[0])

[np.float64(0.0),
 np.float64(0.0),
 np.float64(1.0),
 np.float64(0.0),
 np.float64(1.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(1.0),
 np.float64(1.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(1.0),
 np.float64(0.0),
 np.float64(115.5),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(1.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(1.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(1.0),
 np.float64(0.0),
 np.float64(1.0),
 np.float64(0.0),
 np.float64(1.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(1.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(1.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(1.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(1.0),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(1.0),
 np.float64(72.0),
 np.float64(8425.15)]

In [44]:
#dv.fit(train_dicts)
#dv.transform(train_dicts)

#combining both of the above to a single command

# create a dictionary vectorizer object with the variables oriented as records, 
# which is a list of dictionaries where each dictionary represents a row in the DataFrame.
X_train = dv.fit_transform(train_dicts)
X_train.shape
X_train


array([[0.00000e+00, 0.00000e+00, 1.00000e+00, ..., 1.00000e+00,
        7.20000e+01, 8.42515e+03],
       [1.00000e+00, 0.00000e+00, 0.00000e+00, ..., 0.00000e+00,
        1.00000e+01, 1.02155e+03],
       [1.00000e+00, 0.00000e+00, 0.00000e+00, ..., 0.00000e+00,
        5.00000e+00, 4.13650e+02],
       ...,
       [1.00000e+00, 0.00000e+00, 0.00000e+00, ..., 1.00000e+00,
        2.00000e+00, 1.90050e+02],
       [0.00000e+00, 0.00000e+00, 1.00000e+00, ..., 0.00000e+00,
        2.70000e+01, 7.61950e+02],
       [1.00000e+00, 0.00000e+00, 0.00000e+00, ..., 0.00000e+00,
        9.00000e+00, 7.51650e+02]], shape=(4225, 45))

In [45]:
val_dicts = df_val[categorical + numerical].to_dict(orient = 'Records')
X_val = dv.transform(val_dicts)
X_val.shape

(1409, 45)

## Logistics Regression

## How multiple features become two probabilities

For each customer, logistic regression combines **all their features into one score**, then converts that score into a churn probability.

### 1. Combine the features

After training, the model has learned an intercept and a weight for each feature:

$$
z = b + w_1x_1 + w_2x_2 + \cdots + w_mx_m
$$

- $x_j$: the customer's value for feature $j$.
- $w_j$: the learned weight for that feature.
- $b$: the intercept.
- $z$: the single score produced by combining the features.

### 2. Calculate the two probabilities

Apply the sigmoid to calculate the probability of churn:

$$
P(y=1) = \frac{1}{1+e^{-z}}
$$

Because the customer either stays or churns:

$$
P(y=0) = 1 - P(y=1)
$$

**Example:** if $z=0.90$, the churn probability is approximately $0.711$, and the stay probability is $1-0.711=0.289$.

### 3. Understand the scikit-learn output

```python
probabilities = model.predict_proba(X_val)
```

With `model.classes_` equal to `[0, 1]`, the output columns mean:

| Column index | Outcome | Example probability |
|---|---|---:|
| `0` | Customer stays | 0.289 |
| `1` | Customer churns | 0.711 |

Each row represents one customer, and its probabilities add up to 1.

```python
y_pred = probabilities[:, 1]
```

- `:` selects all customer rows.
- `1` selects the second column: churn probabilities.

**Input columns describe customer features. Output columns describe possible outcomes.** The number of customer rows stays the same.

In [46]:
def sigmoid (z): 
    return 1 / (1 + np.exp(-z))

def linear_regression(xi): 
    result = w0

    for i in range(len(xi)):
        result += w[i] * xi[i]

    return result

def logistic_regression(xi): 
    score = w0

    for j in range(len(xi)):
        score += w[j] * xi[j]
    result = sigmoid(score)
    return result


## Training logistic regression model

In [47]:
from sklearn.linear_model import LogisticRegression

In [48]:
model = LogisticRegression()
model.fit(X_train,y_train)

/usr/local/python/3.14.2/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

In [49]:
## This is the intercept of the logistic regression model, which represents the bias term in the linear equation. 
# It is the value of the log-odds when all features are zero. 
# In other words, it is the baseline log-odds of the positive class (churn) when no other information is provided by the features.

model.intercept_[0]

np.float64(-0.1109287445794238)

In [50]:
# These are the coefficients of the logistic regression model, 
# which represent the weights assigned to each feature in the linear equation.
# the zero indicates that we are accessing the coefficients for the first (and only) class in a binary classification problem.
model.coef_[0].round(3)

array([ 0.564, -0.154, -0.519, -0.013, -0.095,  0.071, -0.098, -0.082,
       -0.033, -0.076, -0.354,  0.343, -0.098,  0.002, -0.251,  0.162,
       -0.02 ,  0.066, -0.098, -0.076,  0.251, -0.098, -0.261, -0.228,
        0.119, -0.121,  0.012, -0.095, -0.03 ,  0.045, -0.028,  0.162,
       -0.271,  0.221, -0.097, -0.098,  0.087, -0.05 , -0.098,  0.039,
        0.194, -0.098, -0.205, -0.072,  0.   ])

In [51]:
# This step uses the trained logistic regression model to make predictions on the training data (X_train).
model.predict(X_train)

array([0, 1, 1, ..., 1, 0, 1], shape=(4225,))

In [52]:
# This is a 2D array where each row corresponds to a sample in the training data, 
# and each column corresponds to a class (0 or 1).
# sum of each row is 1, because the model predicts probabilities for both classes.
model.predict_proba(X_train)

array([[0.91088688, 0.08911312],
       [0.33128828, 0.66871172],
       [0.36229559, 0.63770441],
       ...,
       [0.48776317, 0.51223683],
       [0.96993479, 0.03006521],
       [0.2971009 , 0.7028991 ]], shape=(4225, 2))

In [53]:
# This method returns the predicted probabilities of the positive class (churn) for each instance in the training data.
model.predict_proba(X_train)

model.predict_proba(X_train)[:,1]

# assigning the predicted probabilities of the positive class (churn) to the variable y_pred for the validation data (X_val).
y_pred = model.predict_proba(X_val)[:,1]

# setting a threshold of 0.5 to classify instances as churn (1) or not churn (0) based on the predicted probabilities.
churn_decision = (y_pred > 0.5)


churn_decision

# filtering the validation DataFrame (df_val) to include only the rows where the predicted churn decision is True (1), indicating that the model predicts those customers are likely to churn.
df_val[churn_decision]

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,onlinebackup,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges
3,8433-wxgna,male,0,no,no,2,yes,no,fiber_optic,yes,no,no,no,no,no,month-to-month,yes,electronic_check,75.70,189.20
8,3440-jpscl,female,0,no,no,6,yes,no,fiber_optic,no,no,yes,yes,yes,yes,month-to-month,yes,mailed_check,99.95,547.65
11,2637-fkfsy,female,0,yes,no,3,yes,no,dsl,no,no,no,no,no,no,month-to-month,yes,mailed_check,46.10,130.15
12,7228-omtpn,male,0,no,no,4,yes,no,fiber_optic,no,no,no,no,yes,yes,month-to-month,yes,electronic_check,88.45,370.65
19,6711-fldfb,female,0,no,no,7,yes,yes,fiber_optic,no,no,no,no,no,no,month-to-month,yes,electronic_check,74.90,541.15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1397,5976-jcjrh,male,0,yes,no,10,yes,no,fiber_optic,no,no,no,no,no,no,month-to-month,yes,electronic_check,70.30,738.20
1398,2034-cgrhz,male,1,no,no,24,yes,yes,fiber_optic,no,yes,yes,no,yes,yes,month-to-month,yes,credit_card_(automatic),102.95,2496.70
1399,5276-kqwhg,female,1,no,no,2,yes,no,fiber_optic,no,no,no,no,no,no,month-to-month,yes,electronic_check,69.60,131.65
1407,6521-yytyi,male,0,no,yes,1,yes,yes,fiber_optic,no,no,no,no,yes,yes,month-to-month,yes,electronic_check,93.30,93.30


In [54]:
df_val[churn_decision].customerid

3       8433-wxgna
8       3440-jpscl
11      2637-fkfsy
12      7228-omtpn
19      6711-fldfb
           ...    
1397    5976-jcjrh
1398    2034-cgrhz
1399    5276-kqwhg
1407    6521-yytyi
1408    3049-solay
Name: customerid, Length: 322, dtype: str

In [55]:
## Measuring the performance of the model using accuracy, precision

(y_val == churn_decision.astype(int)).mean()

np.float64(0.801277501774308)

## Model Interpretation

In [56]:
dv.get_feature_names_out()

array(['contract=month-to-month', 'contract=one_year',
       'contract=two_year', 'dependents=no', 'dependents=yes',
       'deviceprotection=no', 'deviceprotection=no_internet_service',
       'deviceprotection=yes', 'gender=female', 'gender=male',
       'internetservice=dsl', 'internetservice=fiber_optic',
       'internetservice=no', 'monthlycharges', 'multiplelines=no',
       'multiplelines=no_phone_service', 'multiplelines=yes',
       'onlinebackup=no', 'onlinebackup=no_internet_service',
       'onlinebackup=yes', 'onlinesecurity=no',
       'onlinesecurity=no_internet_service', 'onlinesecurity=yes',
       'paperlessbilling=no', 'paperlessbilling=yes', 'partner=no',
       'partner=yes', 'paymentmethod=bank_transfer_(automatic)',
       'paymentmethod=credit_card_(automatic)',
       'paymentmethod=electronic_check', 'paymentmethod=mailed_check',
       'phoneservice=no', 'phoneservice=yes', 'seniorcitizen',
       'streamingmovies=no', 'streamingmovies=no_internet_service',

In [57]:
# These are the coefficients of the logistic regression model, 
# which represent the weights assigned to each feature in the linear equation.
# the zero indicates that we are accessing the coefficients for the first (and only) class in a binary classification problem.
model.coef_[0].round(3)

array([ 0.564, -0.154, -0.519, -0.013, -0.095,  0.071, -0.098, -0.082,
       -0.033, -0.076, -0.354,  0.343, -0.098,  0.002, -0.251,  0.162,
       -0.02 ,  0.066, -0.098, -0.076,  0.251, -0.098, -0.261, -0.228,
        0.119, -0.121,  0.012, -0.095, -0.03 ,  0.045, -0.028,  0.162,
       -0.271,  0.221, -0.097, -0.098,  0.087, -0.05 , -0.098,  0.039,
        0.194, -0.098, -0.205, -0.072,  0.   ])

In [58]:
#using the zip function to match the weights to the feature names
a = [1,2,3,4,5]
b = 'abcde'
list(zip(a,b))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd'), (5, 'e')]

In [59]:
list(zip(dv.get_feature_names_out(), model.coef_[0].round(3)))

[('contract=month-to-month', np.float64(0.564)),
 ('contract=one_year', np.float64(-0.154)),
 ('contract=two_year', np.float64(-0.519)),
 ('dependents=no', np.float64(-0.013)),
 ('dependents=yes', np.float64(-0.095)),
 ('deviceprotection=no', np.float64(0.071)),
 ('deviceprotection=no_internet_service', np.float64(-0.098)),
 ('deviceprotection=yes', np.float64(-0.082)),
 ('gender=female', np.float64(-0.033)),
 ('gender=male', np.float64(-0.076)),
 ('internetservice=dsl', np.float64(-0.354)),
 ('internetservice=fiber_optic', np.float64(0.343)),
 ('internetservice=no', np.float64(-0.098)),
 ('monthlycharges', np.float64(0.002)),
 ('multiplelines=no', np.float64(-0.251)),
 ('multiplelines=no_phone_service', np.float64(0.162)),
 ('multiplelines=yes', np.float64(-0.02)),
 ('onlinebackup=no', np.float64(0.066)),
 ('onlinebackup=no_internet_service', np.float64(-0.098)),
 ('onlinebackup=yes', np.float64(-0.076)),
 ('onlinesecurity=no', np.float64(0.251)),
 ('onlinesecurity=no_internet_service

In [60]:
dict(zip(dv.get_feature_names_out(), model.coef_[0].round(3)))

{'contract=month-to-month': np.float64(0.564),
 'contract=one_year': np.float64(-0.154),
 'contract=two_year': np.float64(-0.519),
 'dependents=no': np.float64(-0.013),
 'dependents=yes': np.float64(-0.095),
 'deviceprotection=no': np.float64(0.071),
 'deviceprotection=no_internet_service': np.float64(-0.098),
 'deviceprotection=yes': np.float64(-0.082),
 'gender=female': np.float64(-0.033),
 'gender=male': np.float64(-0.076),
 'internetservice=dsl': np.float64(-0.354),
 'internetservice=fiber_optic': np.float64(0.343),
 'internetservice=no': np.float64(-0.098),
 'monthlycharges': np.float64(0.002),
 'multiplelines=no': np.float64(-0.251),
 'multiplelines=no_phone_service': np.float64(0.162),
 'multiplelines=yes': np.float64(-0.02),
 'onlinebackup=no': np.float64(0.066),
 'onlinebackup=no_internet_service': np.float64(-0.098),
 'onlinebackup=yes': np.float64(-0.076),
 'onlinesecurity=no': np.float64(0.251),
 'onlinesecurity=no_internet_service': np.float64(-0.098),
 'onlinesecurity=yes

In [61]:
small = ['contract', 'tenure', 'monthlycharges']

In [62]:
df_train[small].iloc[:10].to_dict(orient='records')

[{'contract': 'two_year', 'tenure': 72, 'monthlycharges': 115.5},
 {'contract': 'month-to-month', 'tenure': 10, 'monthlycharges': 95.25},
 {'contract': 'month-to-month', 'tenure': 5, 'monthlycharges': 75.55},
 {'contract': 'month-to-month', 'tenure': 5, 'monthlycharges': 80.85},
 {'contract': 'two_year', 'tenure': 18, 'monthlycharges': 20.1},
 {'contract': 'month-to-month', 'tenure': 4, 'monthlycharges': 30.5},
 {'contract': 'month-to-month', 'tenure': 1, 'monthlycharges': 75.1},
 {'contract': 'month-to-month', 'tenure': 1, 'monthlycharges': 70.3},
 {'contract': 'two_year', 'tenure': 72, 'monthlycharges': 19.75},
 {'contract': 'month-to-month', 'tenure': 6, 'monthlycharges': 109.9}]

In [63]:
dicts_train_small = df_train[small].to_dict(orient='records')
dicts_val_small = df_val[small].to_dict(orient='records')

In [64]:
dv_small = DictVectorizer(sparse=False)
dv_small.fit(dicts_train_small)

DictVectorizer(sparse=False)

In [65]:
dv_small.get_feature_names_out()

array(['contract=month-to-month', 'contract=one_year',
       'contract=two_year', 'monthlycharges', 'tenure'], dtype=object)

In [66]:
X_train_small = dv_small.transform(dicts_train_small)
X_train_small

array([[  0.  ,   0.  ,   1.  , 115.5 ,  72.  ],
       [  1.  ,   0.  ,   0.  ,  95.25,  10.  ],
       [  1.  ,   0.  ,   0.  ,  75.55,   5.  ],
       ...,
       [  1.  ,   0.  ,   0.  ,  90.  ,   2.  ],
       [  0.  ,   0.  ,   1.  ,  24.5 ,  27.  ],
       [  1.  ,   0.  ,   0.  ,  80.85,   9.  ]], shape=(4225, 5))

In [67]:
model_small = LogisticRegression()
model_small.fit(X_train_small,y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

In [68]:
model_small.intercept_[0], model_small.coef_[0].round(2)

(np.float64(-2.477957595829565), array([ 0.97, -0.02, -0.95,  0.03, -0.04]))

In [69]:
dict(zip(dv_small.get_feature_names_out(), model_small.coef_[0].round(3)))

{'contract=month-to-month': np.float64(0.971),
 'contract=one_year': np.float64(-0.024),
 'contract=two_year': np.float64(-0.948),
 'monthlycharges': np.float64(0.027),
 'tenure': np.float64(-0.036)}